# 🚀 S4D-RUPA-CLIP: Weakly-Supervised Video Anomaly Detection
### Standalone Residual-to-Text Counterfactual Alignment with S4D Temporal Modeling
**Tác giả:** Huỳnh Quang Vũ (MSSV: 2351010247)  
**Giảng viên hướng dẫn:** TS. Lê Viết Tuấn  
**Trường Đại học Mở TP. Hồ Chí Minh (HCMCOU)**  

---
### 🎯 Mục tiêu khoa học & Hiệu năng:
- **UCF-Crime:** Frame-level ROC-AUC >= 90.0%
- **XD-Violence:** Average Precision (AP) >= 87.0%
- **100% Standalone (Train from Scratch):** Triệt tiêu hoàn toàn DSANet, Dual GCN, Local Transformer và Warm-Start.
- **Cơ chế cốt lõi:**
  1. **Bidirectional S4D Backbone ($O(T)$):** Mô hình hóa ngữ cảnh thời gian tuyến tính hai chiều.
  2. **Normal Prototype Bank ($K=16$):** Tái tạo phông nền bình thường phản trực chứng ($F_{rec}$).
  3. **Residual-to-Text Projector:** Chiếu phần dư bất thường $R = H - F_{rec}$ vào 14 Text Prompts của CLIP. Tách sạch 90% phông nền nhiễu, chỉ giữ lại tín hiệu dị thường!
  4. **Dynamic Evidential Safe Gate:** Điều phối thông minh giữa nhận diện thời gian và phản trực chứng ngữ nghĩa, ngăn chặn triệt để dương tính giả (False Positives).

In [ ]:
# ========================= 1. CLONE & CÀI ĐẶT MÔI TRƯỜNG =========================
import os, sys, shutil, subprocess, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "S4D-RUPA-CLIP"
ARTIFACTS = WORK / "s4d_rupa_artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Clone hoặc Pull code mới nhất từ GitHub
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/Sharvuz/S4D-RUPA-CLIP.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull"], check=False)

os.chdir(str(REPO))

# Cài đặt OpenAI CLIP và các thư viện cần thiết
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ftfy", "regex", "scikit-learn", "pandas", "git+https://github.com/openai/CLIP.git"
], check=True)

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")


## 2. Kiểm tra & Ánh xạ đường dẫn dữ liệu (Auto-Radar)
Tự động quét các Dataset được gắn vào Kaggle (UCF-Crime CLIP features, XD-Violence CLIP features).

In [ ]:
# Quét tự động các file npy và list
import glob
print("Datasets hiện có trong /kaggle/input:")
for d in glob.glob("/kaggle/input/*"):
    print("  -", d)

# Kiểm tra các file ground-truth trong thư mục list/
print("\nGround truth files trong repo list/:")
for f in sorted(glob.glob("list/*")):
    print("  -", f)


## 3. Huấn luyện S4D-RUPA-CLIP trên UCF-Crime (15 Epochs)
- **Backbone:** S4D (Diagonal State Space Model, 2 Layers, d_state=64)
- **Prototypes:** 16 Normal Prototypes
- **Text Prompts:** 14 Multi-Prompt Ensembles
- **Hàm mất mát:** Non-saturating DeepMIL BCE + Residual-to-Text Semantic Alignment

In [ ]:
# ========================= CONFIG: UCF-CRIME =========================
DATASET = "ucf"
MAX_EPOCHS = 15
BATCH_SIZE = 64
LR = 5e-5
WARMUP_EPOCHS = 2
NUM_LAYERS = 2
D_STATE = 64
NUM_PROTOTYPES = 16
SSM_TYPE = "s4d"

# Lưu ý: Thêm -u để in log thời gian thực, không bị đọng buffer
cmd = [
    sys.executable, "-u", "src/ucf_train.py",
    "--dataset", DATASET,
    "--batch-size", str(BATCH_SIZE),
    "--lr", str(LR),
    "--max-epoch", str(MAX_EPOCHS),
    "--warmup-epochs", str(WARMUP_EPOCHS),
    "--num-layers", str(NUM_LAYERS),
    "--d-state", str(D_STATE),
    "--ssm-type", SSM_TYPE,
    "--num-prototypes", str(NUM_PROTOTYPES),
    "--use-text-prompt", "True",
    "--loss-rec-weight", "0.5",
    "--loss-sem-weight", "0.5",
    "--loss-bg-weight", "0.1",
    "--save-dir", "models"
]

print("BẮT ĐẦU HUẤN LUYỆN S4D-RUPA-CLIP TỪ CON SỐ 0 TRÊN UCF-CRIME...")
subprocess.run(cmd, check=True)


## 4. Đánh giá kiểm thử chi tiết trên UCF-Crime Test Set

In [ ]:
test_cmd = [
    sys.executable, "-u", "src/ucf_test.py",
    "--dataset", "ucf",
    "--model-path", "models/s4d_rupa_clip_ucf_best.pth",
    "--num-layers", str(NUM_LAYERS),
    "--d-state", str(D_STATE),
    "--ssm-type", SSM_TYPE,
    "--num-prototypes", str(NUM_PROTOTYPES),
    "--use-text-prompt", "True"
]

print("CHẠY KIỂM THỬ S4D-RUPA-CLIP TRÊN UCF-CRIME...")
subprocess.run(test_cmd, check=True)


## 5. Huấn luyện S4D-RUPA-CLIP trên XD-Violence (20 Epochs)
Mục tiêu bứt phá: Average Precision (AP) >= 87.0%

In [ ]:
# ========================= CONFIG: XD-VIOLENCE =========================
DATASET_XD = "xd"
MAX_EPOCHS_XD = 20
BATCH_SIZE_XD = 64
LR_XD = 5e-5

cmd_xd = [
    sys.executable, "-u", "src/xd_train.py",
    "--dataset", DATASET_XD,
    "--batch-size", str(BATCH_SIZE_XD),
    "--lr", str(LR_XD),
    "--max-epoch", str(MAX_EPOCHS_XD),
    "--warmup-epochs", "2",
    "--num-layers", "2",
    "--d-state", "64",
    "--ssm-type", "s4d",
    "--num-prototypes", "16",
    "--use-text-prompt", "True",
    "--save-dir", "models"
]

print("BẮT ĐẦU HUẤN LUYỆN S4D-RUPA-CLIP TỪ CON SỐ 0 TRÊN XD-VIOLENCE...")
subprocess.run(cmd_xd, check=True)


## 6. Lưu trữ Checkpoints & Đóng gói Artifacts

In [ ]:
# Sao chép các checkpoint tốt nhất sang thư mục artifacts để tải về
for ckpt in glob.glob("models/*.pth"):
    dst = ARTIFACTS / os.path.basename(ckpt)
    shutil.copy(ckpt, dst)
    print(f"Đã lưu: {dst} ({os.path.getsize(dst) / 1e6:.2f} MB)")

print(f"\nArtifacts đã sẵn sàng tại: {ARTIFACTS}")
